# PILOT STUDY (negative result): CATAN Cities & Knights rulebook

> **This notebook is a preserved pilot, not the primary experiment.** On this 20-page, 98-chunk corpus the Cross-Encoder produced
> **0 promotions and 0 demotions over 19 queries** (see Stage 5). The primary experiment uses NIST SP 800-30 Rev. 1 in
> `RAG_Baseline_vs_Rerank.ipynb`. **No metric from this pilot is combined with the NIST results.**

# RAG Assignment: Baseline vs Re-Ranking (pilot corpus)
**Corpus:** *CATAN: Cities & Knights* game rules & almanac (2020 edition, official PDF from catan.com)

This notebook is built stage by stage. **This part covers Stages 1-2 only:**

1. PDF verification, text extraction and cleaning (block/column aware)
2. Chunking with metadata, then **freezing** the corpus

Embeddings, retrieval, Cross-Encoder re-ranking, generation and the experiments are added in later stages.

**Reproducibility note.** The rulebook is copyright Catan GmbH and the PDF carries a publisher "no-copy"
permission flag. The PDF and the extracted chunk text are therefore **not committed** to Git (`data/` is
git-ignored). The notebook fetches the PDF from its official URL and refuses to continue unless its SHA-256
matches. Only a text-free manifest (`results/corpus_manifest.json`: chunk ids, pages, sections, token
counts and text hashes) is committed.

In [1]:
import collections
import hashlib
import json
import os
import re
import statistics
import urllib.request
from pathlib import Path

# Keep every downloaded model file inside the (git-ignored) project folder, not in ~/.cache. Must be set BEFORE
# huggingface_hub / sentence_transformers are imported.
os.environ.setdefault("HF_HOME", str(Path("data/hf_cache").resolve()))

# KNOWN ISSUE (found in Stage 3): on macOS, torch, faiss-cpu and scikit-learn each bundle their own libomp.dylib. With more
# than one OpenMP thread the kernel dies with SIGSEGV inside a libomp worker thread (reproduced with either import order).
# One OpenMP thread creates no worker threads and avoids the clash. This must be set BEFORE torch/faiss are imported, and it
# means every timing measured later in this notebook is single-threaded CPU (identical for both pipelines).
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pymupdf
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

# --- source document -------------------------------------------------------------------------------------
PDF_URL = "https://www.catan.com/sites/default/files/2021-06/catan_c_k_2020_rule_book_200708.pdf"
PDF_SHA256 = "ae335149a56b60418b5561cf9f45dea07cdd00be01362a1e51802f7bf2f024ac"
PDF_PATH = Path("data/catan_pilot/catan_ck_2020_rules.pdf")

# --- embedding model whose tokenizer measures chunk length (the model itself is used in Stage 3) -----------
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# --- the frozen corpus: any change to extraction/chunking that alters a single chunk makes the notebook stop ---
FROZEN_CORPUS_SHA256 = "a3a1772d90e3a71bcacf6f1b5a70e01dc7237f4e5def64b7381eeb1877c61950"

# --- chunking parameters (FROZEN before any evaluation query is written) ---------------------------------
MAX_CHUNK_TOKENS = 200     # soft cap per chunk, counted with the embedding model's tokenizer
HARD_LIMIT_TOKENS = 240    # never exceed this; stays under the embedding model's own truncation limit
MIN_TOKENS = 30            # avoid a tiny trailing chunk inside a split section
OVERLAP_SENTENCES = 1      # sentences repeated at the start of a continuation chunk
MAX_OVERLAP_TOKENS = 40    # skip the overlap if that sentence is longer than this
MIN_OVERLAP_TOKENS = 8     # ...or shorter than this (e.g. "See Illustration J." carries no information)
ENTRY_MAX_TOKENS = 240     # an Almanac card entry is atomic: allow it up to the hard limit instead of splitting it

# --- pages / sections that are not rules content ----------------------------------------------------------
EXCLUDE_PAGES = {1, 19}    # 1 = cover art, 19 = dot-leader page index + card-sleeve advert
EXCLUDE_HEADINGS = {"credits"}
BOILERPLATE_RE = re.compile(
    r"(Original Development:|Development Team:|^Art:|Art Direction|^Production:|Special Thanks|"
    r"\"Catan Sun\"|“Catan Sun”|under license by|Published by Catan Studio|Gamegenic|"
    r"components missing or damaged|catanstudio\.com|klausteuber\.com|For all other inquiries|"
    r"further information online|info@)", re.I)

OUT_DIR = Path("pilot_catan/results")
DATA_DIR = Path("data/catan_pilot")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

/Users/charan/Documents/LanguageModels&Agents/RAGAssignment/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Stage 1a. Fetch and verify the PDF
The hash check guarantees every run (and the professor's run) uses byte-identical input, so chunk ids are stable.

In [2]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def ensure_pdf() -> Path:
    """Download the official PDF if absent, then verify its SHA-256 (fail loudly on mismatch)."""
    if not PDF_PATH.exists():
        print("Downloading", PDF_URL)
        urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    got = sha256_file(PDF_PATH)
    assert got == PDF_SHA256, f"PDF hash mismatch: expected {PDF_SHA256}, got {got}"
    return PDF_PATH


ensure_pdf()
doc = pymupdf.open(PDF_PATH)
print(f"PDF verified: {PDF_PATH}  sha256={PDF_SHA256[:16]}...  pages={len(doc)}")

PDF verified: data/catan_pilot/catan_ck_2020_rules.pdf  sha256=ae335149a56b6041...  pages=20


## Stage 1b. Tokenizer used to measure chunk length
Chunk size is measured in *word-pieces of the embedding model's own tokenizer*, because the Bi-Encoder silently
truncates anything beyond its maximum sequence length. We read that limit from the model's config instead of assuming it.

In [3]:
tok_file = hf_hub_download(EMBED_MODEL, "tokenizer.json")
cfg_file = hf_hub_download(EMBED_MODEL, "sentence_bert_config.json")
MODEL_MAX_SEQ = json.load(open(cfg_file))["max_seq_length"]
tok = Tokenizer.from_file(tok_file)
tok.no_truncation()
tok.no_padding()


def n_tokens(text: str) -> int:
    """Token count including [CLS]/[SEP], i.e. what the Bi-Encoder actually sees."""
    return len(tok.encode(text).ids)


assert HARD_LIMIT_TOKENS <= MODEL_MAX_SEQ, "chunks must fit in the embedding model's window"
print(f"{EMBED_MODEL}: max_seq_length={MODEL_MAX_SEQ} tokens; our hard limit={HARD_LIMIT_TOKENS}, soft cap={MAX_CHUNK_TOKENS}")

sentence-transformers/all-MiniLM-L6-v2: max_seq_length=256 tokens; our hard limit=240, soft cap=200


## Stage 1c. Block- and column-aware extraction
Plain `pdftotext`/`get_text()` order interleaves illustration labels into body paragraphs (e.g. page 5, the robber rule)
and detaches Almanac headings from their bodies. Instead we read every text *line* with its position and font and:

* **drop non-content lines**: tiny text drawn inside card artwork (<10.5 pt), copyright marks, page numbers,
  map numbers, display/illustration captions, and table labels sitting on top of illustrations;
* **order lines by layout**: full-width lines split the page into bands; inside a band the left column is read
  before the right column;
* **rebuild paragraphs** from line spacing, bullets and explicit "Example:/Note:" markers, and detect headings by font.

Every dropped line is logged with its reason so the loss can be audited.

In [4]:
COL_SPLIT_X = 295.0                      # page is 594 pt wide: left column x0 < 295 <= right column x0
NOTE_MARKERS = ("•", "Example:", "Note:", "Notes:", "Important:", "Reminder:", "Tip:")


def line_info(line, illustration_boxes):
    """Return a dict for one text line, with `why` set if the line must be dropped."""
    spans = [s for s in line["spans"] if s["text"].strip()]
    if not spans:
        return None
    text = "".join(s["text"] for s in line["spans"]).strip()
    n_chars = sum(len(s["text"]) for s in spans)
    size = max(s["size"] for s in spans)
    font = max(spans, key=lambda s: len(s["text"]))["font"]
    italic = sum(len(s["text"]) for s in spans if s["flags"] & 2) / n_chars > 0.6
    x0, y0, x1, y1 = line["bbox"]
    centre = pymupdf.Point((x0 + x1) / 2, (y0 + y1) / 2)
    why = None
    if size < 10.5:
        why = "small text (card artwork / credits / contact)"
    elif font.startswith("ArialMT") or "Catan GmbH" in text:
        why = "copyright mark"
    elif font.startswith(("Catan-2015", "Ogham", "Palatino")):
        why = "page number / logo / decoration / map number"
    elif font.startswith("AGaramondPro") and not (font.startswith("AGaramondPro-Bold-SC800") and size >= 30):
        why = "display text / illustration caption"
    elif any(b.contains(centre) for b in illustration_boxes) and not (italic and abs(size - 11.0) < 0.05) \
            and not (font.startswith("AGaramondPro-Bold-SC800") and size >= 30):
        why = "label on top of an illustration"   # italic 11 pt body text that overlaps an image box is kept
    banner = font.startswith("AGaramondPro-Bold-SC800") and size >= 30      # Almanac category banner
    return dict(text=text, size=size, font=font, italic=italic, x0=x0, y0=y0, x1=x1, y1=y1, why=why, banner=banner)


def is_heading(L):
    f = L["font"]
    return ("BdCond" in f and L["size"] >= 12.5) or ("BkCond" in f and not L["italic"] and 12.5 <= L["size"] < 14.5)


def order_lines(lines):
    """Reading order: full-width lines delimit bands; inside a band, left column first, then right column."""
    lines = sorted(lines, key=lambda L: (L["y0"], L["x0"]))
    ordered, band = [], []

    def flush():
        nonlocal band
        left = sorted([L for L in band if L["x0"] < COL_SPLIT_X], key=lambda L: L["y0"])
        right = sorted([L for L in band if L["x0"] >= COL_SPLIT_X], key=lambda L: L["y0"])
        ordered.extend(left + right)
        band = []

    for L in lines:
        if L["x1"] - L["x0"] > 300 and L["x0"] < 100:      # full-width line
            flush()
            ordered.append(L)
        else:
            band.append(L)
    flush()
    return ordered


def build_paragraphs(ordered):
    paras, cur, prev = [], None, None
    for L in ordered:
        kind = "banner" if L["banner"] else ("heading" if is_heading(L) else "text")
        new = cur is None or kind != cur["kind"]
        if not new and kind == "heading":
            new = abs(L["size"] - prev["size"]) > 0.5 or L["y0"] - prev["y0"] > 1.6 * L["size"]
        elif not new and kind == "text":
            pitch = L["y0"] - prev["y0"]
            new = pitch < -1 or pitch > 1.2 * L["size"] + 1.0 or L["text"].startswith(NOTE_MARKERS)
        if new:
            cur = dict(kind=kind, lines=[L], size=L["size"])
            paras.append(cur)
        else:
            cur["lines"].append(L)
        prev = L
    return paras


def clean_text(t: str) -> str:
    t = re.sub(r"\s+", " ", t.replace("\t", " ")).strip()
    t = re.sub(r"(?<=\w)/ (?=[a-z])", "/", t)           # "and/ or"        -> "and/or"
    t = re.sub(r"(?<=[a-z])– (?=[a-z])", "–", t)         # "politics– blue" -> "politics–blue"
    t = re.sub(r"(?<=[a-z])- (?=[a-z])", "-", t)         # "flip- chart"    -> "flip-chart"
    return t


def extract_document(doc):
    """Return (paragraphs in reading order across pages, audit log of dropped lines)."""
    all_paras, dropped = [], []
    for pno, page in enumerate(doc, 1):
        d = page.get_text("dict")
        page_area = page.rect.width * page.rect.height
        boxes = [pymupdf.Rect(b["bbox"]) for b in d["blocks"]
                 if b["type"] == 1 and pymupdf.Rect(b["bbox"]).get_area() < 0.8 * page_area]   # skip full-page background
        keep = []
        for b in d["blocks"]:
            if b["type"] != 0:
                continue
            for l in b["lines"]:
                L = line_info(l, boxes)
                if L is None:
                    continue
                if pno in EXCLUDE_PAGES:
                    dropped.append(dict(page=pno, reason="excluded page", text=L["text"]))
                elif L["why"]:
                    dropped.append(dict(page=pno, reason=L["why"], text=L["text"]))
                else:
                    keep.append(L)
        for p in build_paragraphs(order_lines(keep)):
            p["text"] = clean_text(" ".join(L["text"] for L in p["lines"]))
            p["page"] = pno
            p["level"] = 1 if p["size"] >= 14.5 else 2
            all_paras.append(p)
    return all_paras, dropped


paragraphs, dropped_lines = extract_document(doc)

# --- post-filters: credits / legal / contact boilerplate -------------------------------------------------------
content, excluded_paras = [], []
skip_until_heading = False
for p in paragraphs:
    if p["kind"] == "heading":
        skip_until_heading = p["text"].strip().lower() in EXCLUDE_HEADINGS
        if skip_until_heading:
            excluded_paras.append(p)
            continue
    if skip_until_heading or (p["kind"] == "text" and BOILERPLATE_RE.search(p["text"])):
        excluded_paras.append(p)
        continue
    content.append(p)

# --- stitch paragraphs that were cut by a page or column break in the middle of a sentence ------------------------
stitched, n_stitched = [], 0
for p in content:
    prev = stitched[-1] if stitched else None
    if (prev and prev["kind"] == "text" and p["kind"] == "text" and p["text"][:1].islower()
            and not prev["text"].rstrip().endswith((".", "!", "?", ":", "”", ")", '"'))):
        prev["text"] += " " + p["text"]
        n_stitched += 1
    else:
        stitched.append(p)
content = stitched
print(f"stitched {n_stitched} paragraph(s) that were split mid-sentence by a page/column break")

stitched 3 paragraph(s) that were split mid-sentence by a page/column break


In [5]:
raw_words = sum(len(pg.get_text().split()) for pg in doc)
drop_by_reason = collections.Counter()
for d in dropped_lines:
    drop_by_reason[d["reason"]] += len(d["text"].split())
excl_words = sum(len(p["text"].split()) for p in excluded_paras)
kept_words = sum(len(p["text"].split()) for p in content if p["kind"] != "banner")
print(f"PDF pages: {len(doc)}   raw words (PyMuPDF get_text, all pages): {raw_words:,}")
print("Words dropped by line filters:")
for reason, n in drop_by_reason.most_common():
    print(f"   {n:6,d}  {reason}")
print(f"Words dropped as credits/legal/contact paragraphs: {excl_words:,}")
print(f"Words kept as content (headings + body): {kept_words:,}   paragraphs: {sum(1 for p in content if p['kind']!='banner'):,}")

PDF pages: 20   raw words (PyMuPDF get_text, all pages): 13,328
Words dropped by line filters:
    2,072  small text (card artwork / credits / contact)
    1,185  excluded page
      108  label on top of an illustration
       78  display text / illustration caption
       45  page number / logo / decoration / map number
       20  copyright mark
Words dropped as credits/legal/contact paragraphs: 167
Words kept as content (headings + body): 9,637   paragraphs: 330


## Stage 1d. Audit: the problematic rule page (page 5) after cleaning
Plain extraction spliced the "Cities Produce Resource & Commodity Cards" illustration labels into the robber rule.
Below is the same paragraph after layout-aware cleaning (excerpt only).

In [6]:
def show_paragraphs(page, contains=None, max_words=140):
    for p in content:
        if p["page"] == page and p["kind"] == "text" and (contains is None or contains in p["text"]):
            words = p["text"].split()
            print(" ".join(words[:max_words]) + (" ..." if len(words) > max_words else ""))
            print()


print("--- page 5, robber paragraph (cleaned) ---")
show_paragraphs(5, contains="occupied by the robber", max_words=170)

--- page 5, robber paragraph (cleaned) ---
As in Catan base game, any hex that is occupied by the robber does not produce any resources or commodities. Also, if a “7” is rolled on the dice, no hexes produce: instead, the player who rolled the “7” must move the robber and may be able to steal 1 card from any opponent whose settlement or city is adjacent to the robber. The robber cannot be moved until after the barbarians reach the island of Catan for the first time. Until the first barbarian attack, if you roll a “7”, all players must check if they are holding too many cards as usual; however, you do not move the robber from the desert and you do not steal a card from another player. Further, you are not allowed to move the robber by playing of any progress cards (like the Bishop) or knight actions until after the first barbarian attack.



## Stage 2a. Chunking (structure-aware, frozen parameters)
* A **section** = text under one heading (a rules sub-heading, or one Almanac card entry such as *Smith (2)*).
  **Chunks never cross a heading**, so an Almanac entry is never glued to its neighbour.
* Paragraphs are packed greedily up to `MAX_CHUNK_TOKENS`. A paragraph is only split (at sentence boundaries)
  if it alone exceeds the cap. The heading text stays at the start of the section's first chunk, as in the document.
* When a section continues into a second chunk, the last sentence of the previous chunk is repeated (overlap),
  unless it is long. This keeps a clause that straddles a boundary readable in one chunk.
* Metadata (not embedded): `chunk_id`, pages, `section`, Almanac `entry` and `category`, token count, text hash.

In [7]:
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])[\"”’)]?\s+(?=[\"“(•A-Z0-9])")


def split_sentences(text: str):
    parts, start = [], 0
    for m in SENT_SPLIT_RE.finditer(text):
        parts.append(text[start:m.end()].strip())
        start = m.end()
    parts.append(text[start:].strip())
    return [s for s in parts if s]


def split_oversized(block: str, cap: int):
    """Split a block that alone exceeds the cap into sentence groups that each fit."""
    pieces, cur = [], []
    for s in split_sentences(block):
        if cur and n_tokens(" ".join(cur + [s])) > cap:
            pieces.append(" ".join(cur))
            cur = []
        cur.append(s)
    if cur:
        pieces.append(" ".join(cur))
    return pieces


def chunk_section(blocks, cap=MAX_CHUNK_TOKENS):
    """Pack blocks of ONE section into chunks of at most ~`cap` tokens. Returns list of (text, has_overlap)."""
    pieces = []
    for b in blocks:
        pieces += [b] if n_tokens(b) <= cap else split_oversized(b, cap)
    chunks, cur, cur_overlap = [], [], False
    for i, piece in enumerate(pieces):
        candidate = "\n".join(cur + [piece])
        if cur and n_tokens(candidate) > cap:
            remaining = n_tokens("\n".join(pieces[i:]))
            if remaining < MIN_TOKENS and n_tokens(candidate + "\n" + "\n".join(pieces[i + 1:])) <= HARD_LIMIT_TOKENS:
                cur.append(piece)                 # absorb a tiny tail instead of emitting a stub chunk
                continue
            chunks.append(("\n".join(cur), cur_overlap))
            last_sentence = split_sentences(cur[-1])[-1] if OVERLAP_SENTENCES else ""
            if last_sentence and MIN_OVERLAP_TOKENS <= n_tokens(last_sentence) <= MAX_OVERLAP_TOKENS and \
                    n_tokens(last_sentence + "\n" + piece) <= cap:
                cur, cur_overlap = [last_sentence, piece], True
            else:
                cur, cur_overlap = [piece], False
        else:
            cur.append(piece)
    if cur:
        chunks.append(("\n".join(cur), cur_overlap))
    return chunks


def build_chunks(content_paras):
    h1, h2, category = "Introduction", None, None
    sections = []                    # each: dict(h1,h2,category,heads,blocks,pages)
    cur = None
    pending_heads = []
    for p in content_paras:
        if p["kind"] == "banner":
            category = clean_text(" ".join(L["text"] for L in p["lines"])).split()[0]   # "Science Cards" -> "Science"
            continue
        if p["kind"] == "heading":
            if p["level"] == 1:
                h1, h2 = p["text"], None
            else:
                h2 = p["text"]
            pending_heads.append((p["text"], p["page"]))
            cur = None
            continue
        if cur is None:
            cur = dict(h1=h1, h2=h2, category=category if (h1 or "").lower().startswith("the almanac") else None,
                       blocks=[], pages=set())
            for t, pg in pending_heads:
                cur["pages"].add(pg)
            cur["heads"] = [t for t, _ in pending_heads]
            pending_heads = []
            sections.append(cur)
        cur["blocks"].append(p["text"])
        cur["pages"].add(p["page"])
    chunks = []
    for sec in sections:
        blocks = list(sec["blocks"])
        blocks[0] = "\n".join(sec["heads"] + [blocks[0]]) if sec["heads"] else blocks[0]   # heading stays in the text
        section_name = " > ".join(x for x in (sec["h1"], sec["h2"]) if x)
        is_entry = (sec["h1"] or "").lower().startswith("the almanac") and sec["h2"]
        for text, has_overlap in chunk_section(blocks, ENTRY_MAX_TOKENS if is_entry else MAX_CHUNK_TOKENS):
            chunks.append(dict(
                text=text, section=section_name, entry=sec["h2"] if is_entry else None, category=sec["category"],
                page_start=min(sec["pages"]), page_end=max(sec["pages"]), has_overlap=has_overlap))
    for i, c in enumerate(chunks, 1):
        c["chunk_id"] = f"ck-{i:03d}"
        c["n_tokens"] = n_tokens(c["text"])
        c["text_sha256"] = hashlib.sha256(c["text"].encode("utf-8")).hexdigest()
    return chunks


CHUNKS = build_chunks(content)

## Stage 2b. Chunking statistics and invariants

In [8]:
toks = [c["n_tokens"] for c in CHUNKS]
rules_chunks = [c for c in CHUNKS if c["entry"] is None]
almanac_chunks = [c for c in CHUNKS if c["entry"] is not None]
entries = collections.Counter(c["entry"] for c in almanac_chunks)
split_entries = [e for e, n in entries.items() if n > 1]
pcts = statistics.quantiles(toks, n=10)
print(f"chunks: {len(CHUNKS)}  (rules/set-up sections: {len(rules_chunks)}, Almanac entries: {len(almanac_chunks)})")
print(f"tokens/chunk  min={min(toks)}  median={statistics.median(toks):.0f}  mean={statistics.mean(toks):.0f}  "
      f"p90={pcts[-1]:.0f}  max={max(toks)}   (soft cap {MAX_CHUNK_TOKENS}, hard limit {HARD_LIMIT_TOKENS}, model window {MODEL_MAX_SEQ})")
print(f"chunks under {MIN_TOKENS} tokens: {sum(t < MIN_TOKENS for t in toks)}   "
      f"chunks over soft cap: {sum(t > MAX_CHUNK_TOKENS for t in toks)}   chunks with overlap prefix: {sum(c['has_overlap'] for c in CHUNKS)}")
print(f"Almanac entries: {len(entries)}; entries split across >1 chunk: {split_entries or 'none'}")
print(f"words in chunks (incl. repeated overlap sentences): {sum(len(c['text'].split()) for c in CHUNKS):,}")
print("chunks per page:", dict(sorted(collections.Counter(c["page_start"] for c in CHUNKS).items())))

# Invariants the later stages rely on
assert max(toks) <= HARD_LIMIT_TOKENS <= MODEL_MAX_SEQ, "a chunk would be truncated by the Bi-Encoder"
assert len({c["chunk_id"] for c in CHUNKS}) == len(CHUNKS)
assert all(c["text"].strip() for c in CHUNKS)
print("invariants OK: unique ids, no empty chunk, every chunk fits the Bi-Encoder window")

chunks: 98  (rules/set-up sections: 73, Almanac entries: 25)
tokens/chunk  min=25  median=120  mean=125  p90=190  max=227   (soft cap 200, hard limit 240, model window 256)
chunks under 30 tokens: 2   chunks over soft cap: 4   chunks with overlap prefix: 23
Almanac entries: 25; entries split across >1 chunk: none
words in chunks (incl. repeated overlap sentences): 10,023
chunks per page: {2: 5, 3: 7, 4: 3, 5: 6, 6: 8, 7: 7, 8: 3, 9: 8, 10: 7, 11: 6, 12: 3, 13: 8, 14: 6, 15: 4, 16: 5, 17: 7, 18: 3, 20: 2}
invariants OK: unique ids, no empty chunk, every chunk fits the Bi-Encoder window


## Stage 2c. Audit: Almanac entries Smith (2) and Printer (1)
Headings and bodies are intact and complete, each in a single chunk.

In [9]:
def show_chunk(c):
    print(f"[{c['chunk_id']}] pages {c['page_start']}-{c['page_end']} | section: {c['section']} | "
          f"category: {c['category']} | {c['n_tokens']} tokens | overlap: {c['has_overlap']}")
    print(c["text"])
    print()


for c in CHUNKS:
    if c["entry"] in ("Smith (2)", "Printer (1)"):
        show_chunk(c)

[ck-079] pages 15-15 | section: The Almanac > Printer (1) | category: Science | 72 tokens | overlap: False
Printer (1)
This card awards you 1 victory point.
You must play this card face up in front of you immediately when you draw it, even if it is not your turn. You may not keep VP cards in your hand. It does not count toward your hand limit of 4 progress cards, and it may not be stolen by the Spy.

[ck-081] pages 15-15 | section: The Almanac > Smith (2) | category: Science | 96 tokens | overlap: False
Smith (2)
You may promote 2 of your knights 1 level each for free.
You may not promote “mighty” knights. You may promote a knight regardless of whether it is active or inactive. The condition of the knight (active or inactive) does not change. You may only promote a “strong” knight (level 2) if you have built the “Fortress” city improvement (politics–blue). You may only promote a knight one level per turn.



## Stage 2d. Freeze the corpus
From here on **both pipelines read the same `CHUNKS` object** loaded from disk, and `assert_frozen()` re-checks a
SHA-256 over every (chunk_id, text) pair. Reranking is therefore the only thing that can differ between the pipelines.

In [10]:
def corpus_hash(chunks) -> str:
    h = hashlib.sha256()
    for c in chunks:
        h.update(c["chunk_id"].encode() + b"\x00" + c["text"].encode("utf-8") + b"\x01")
    return h.hexdigest()


CORPUS_SHA256 = corpus_hash(CHUNKS)
assert CORPUS_SHA256 == FROZEN_CORPUS_SHA256, "extraction/chunking changed: the corpus is no longer the frozen one"
with open(DATA_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:       # contains rulebook text -> git-ignored
    for c in CHUNKS:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

manifest = dict(
    source=dict(title="CATAN: Cities & Knights - Game Rules & Almanac (3-4 players, 2020)", url=PDF_URL, sha256=PDF_SHA256,
                pages=len(doc)),
    parameters=dict(embed_model_tokenizer=EMBED_MODEL, model_max_seq_length=MODEL_MAX_SEQ, max_chunk_tokens=MAX_CHUNK_TOKENS,
                    hard_limit_tokens=HARD_LIMIT_TOKENS, min_tokens=MIN_TOKENS, overlap_sentences=OVERLAP_SENTENCES,
                    exclude_pages=sorted(EXCLUDE_PAGES), extractor=f"PyMuPDF {pymupdf.__version__}"),
    corpus_sha256=CORPUS_SHA256, n_chunks=len(CHUNKS),
    chunks=[{k: c[k] for k in ("chunk_id", "section", "entry", "category", "page_start", "page_end", "n_tokens",
                               "has_overlap", "text_sha256")} for c in CHUNKS])
(OUT_DIR / "corpus_manifest.json").write_text(json.dumps(manifest, indent=1, ensure_ascii=False))


def load_frozen_chunks():
    """What every pipeline calls: load chunks from disk and verify they are the frozen corpus."""
    chunks = [json.loads(line) for line in open(DATA_DIR / "chunks.jsonl", encoding="utf-8")]
    expected = json.load(open(OUT_DIR / "corpus_manifest.json"))["corpus_sha256"]
    assert corpus_hash(chunks) == expected, "corpus changed since it was frozen"
    return chunks


assert load_frozen_chunks() == CHUNKS              # round-trip test
print(f"frozen corpus: {len(CHUNKS)} chunks   corpus_sha256={CORPUS_SHA256}")
print("manifest written to results/corpus_manifest.json (no rulebook text); chunks written to data/chunks.jsonl (git-ignored)")

frozen corpus: 98 chunks   corpus_sha256=a3a1772d90e3a71bcacf6f1b5a70e01dc7237f4e5def64b7381eeb1877c61950
manifest written to results/corpus_manifest.json (no rulebook text); chunks written to data/chunks.jsonl (git-ignored)


## Stage 2e. Candidate evaluation questions and their gold evidence chunks
Gold chunks are located by **exact evidence phrases** and asserted to exist, so the labels cannot silently drift.
These are *candidate* questions: no retrieval has been run yet, so nothing here says the baseline fails on them.

In [11]:
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", s.replace("“", '"').replace("”", '"').replace("’", "'")).strip().lower()


def find_chunks(*phrases):
    return [c for c in CHUNKS if all(norm(p) in norm(c["text"]) for p in phrases)]


CANDIDATES = {
    "Q1 robber before the first barbarian attack": dict(
        question="If I roll a 7 before the barbarians have attacked for the first time, can I move the robber?",
        evidence=[("The robber cannot be moved until after the barbarians reach the island of Catan for the first time",
                   "you do not move the robber from the desert")]),
    "Q2 Smith card and mighty knights": dict(
        question="Can the Smith progress card be used to promote a mighty knight?",
        evidence=[("You may not promote \"mighty\" knights", "Fortress")]),
    "Q3 (control) VP cards and the hand limit": dict(
        question="Do Victory Point progress cards count toward the 4-card progress-card hand limit?",
        evidence=[("does not count toward your hand limit of 4 progress cards",),
                  ("do not count toward your hand limit of 4 cards",)]),
}
for name, q in CANDIDATES.items():
    print(f"{name}\n  question: {q['question']}")
    gold = []
    for ev in q["evidence"]:
        found = find_chunks(*ev)
        assert found, f"evidence not found in any chunk: {ev}"
        gold += found
    q["gold_chunk_ids"] = sorted({c["chunk_id"] for c in gold})
    for c in {c["chunk_id"]: c for c in gold}.values():
        print(f"  gold chunk {c['chunk_id']} | page {c['page_start']} | {c['section']} | {c['n_tokens']} tokens")
    print()
print("All gold evidence phrases located. (Labels will be re-verified by you before any experiment is run.)")

Q1 robber before the first barbarian attack
  question: If I roll a 7 before the barbarians have attacked for the first time, can I move the robber?
  gold chunk ck-021 | page 5 | Specific Actions > Dice Rolls | 180 tokens

Q2 Smith card and mighty knights
  question: Can the Smith progress card be used to promote a mighty knight?
  gold chunk ck-081 | page 15 | The Almanac > Smith (2) | 96 tokens

Q3 (control) VP cards and the hand limit
  question: Do Victory Point progress cards count toward the 4-card progress-card hand limit?
  gold chunk ck-079 | page 15 | The Almanac > Printer (1) | 72 tokens
  gold chunk ck-083 | page 16 | The Almanac > Constitution (1) | 72 tokens
  gold chunk ck-043 | page 9 | Progress Cards > Playing Progress Cards | 196 tokens

All gold evidence phrases located. (Labels will be re-verified by you before any experiment is run.)


---
# Stage 3. Embeddings and vector index
Scope: embed the **frozen** chunks with the Bi-Encoder and build the cosine index. **No question is run in this stage**
(retrieval is Stage 4). Checks performed here:
1. the loaded model's real `max_seq_length`, and direct proof that no chunk is silently truncated;
2. unit-length vectors, so inner product == cosine;
3. an index smoke test that uses no evaluation question;
4. flags for repeated set-up text and overlapping neighbours (an analysis aid only; the corpus is **not** changed).

**Known issue recorded here:** the first run of this stage crashed the kernel (SIGSEGV in `libomp.dylib`) because torch,
faiss-cpu and scikit-learn each ship their own OpenMP runtime. The fix is `OMP_NUM_THREADS=1`, set in the first cell.
Consequence: all CPU timings in this notebook are single-threaded.

In [12]:
import importlib.metadata as md
import platform
import time

import faiss
import numpy as np
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer

EMBED_DEVICE = "cpu"          # fixed on purpose: reproducible numbers and fair latency measurements later
INDEX_PATH = DATA_DIR / "index.faiss"
EMB_PATH = DATA_DIR / "embeddings.npy"

VERSIONS = {"python": platform.python_version(), "machine": platform.machine()}
for pkg in ("torch", "sentence-transformers", "transformers", "faiss-cpu", "numpy", "pymupdf", "tokenizers"):
    VERSIONS[pkg] = md.version(pkg)
print(json.dumps(VERSIONS, indent=1))
print("faiss module version:", faiss.__version__, "| torch threads:", torch.get_num_threads(), "| faiss OpenMP threads:", faiss.omp_get_max_threads())
assert os.environ["OMP_NUM_THREADS"] == "1" and torch.get_num_threads() == 1, "OpenMP workaround not active"

W1018 00:26:38.246000 33106 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


{
 "python": "3.14.7",
 "machine": "arm64",
 "torch": "2.14.1",
 "sentence-transformers": "6.1.0",
 "transformers": "5.20.1",
 "faiss-cpu": "1.15.1",
 "numpy": "2.5.4",
 "pymupdf": "1.28.2",
 "tokenizers": "0.23.3"
}
faiss module version: 1.15.1 | torch threads: 1 | faiss OpenMP threads: 1


In [13]:
CHUNKS = load_frozen_chunks()                                     # re-verifies the hash stored in the manifest
HASH_BEFORE = corpus_hash(CHUNKS)
FILE_HASH_BEFORE = sha256_file(DATA_DIR / "chunks.jsonl")
assert HASH_BEFORE == FROZEN_CORPUS_SHA256
print(f"[before] {len(CHUNKS)} frozen chunks | corpus_sha256 = {HASH_BEFORE}")
print(f"[before] data/chunks.jsonl file sha256 = {FILE_HASH_BEFORE}")

[before] 98 frozen chunks | corpus_sha256 = a3a1772d90e3a71bcacf6f1b5a70e01dc7237f4e5def64b7381eeb1877c61950
[before] data/chunks.jsonl file sha256 = 489f7f204881460bafc493da8b29a674fdba192291595c05b1d3210f75c032e5


In [14]:
torch.manual_seed(0)
model = SentenceTransformer(EMBED_MODEL, device=EMBED_DEVICE)
DIM = model.get_sentence_embedding_dimension()
print(model)
print(f"loaded {EMBED_MODEL} on {EMBED_DEVICE}: embedding dim={DIM}, model.max_seq_length={model.max_seq_length}")
assert model.max_seq_length == MODEL_MAX_SEQ, "loaded model disagrees with the config read in Stage 1b"

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19141.04it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
loaded sentence-transformers/all-MiniLM-L6-v2 on cpu: embedding dim=384, model.max_seq_length=256


/var/folders/nc/mwdjtcy57ss8w8gfjngyvjzh0000gn/T/ipykernel_33106/3022795274.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = model.get_sentence_embedding_dimension()


In [15]:
texts = [c["text"] for c in CHUNKS]
full_len = [len(model.tokenizer(t, truncation=False)["input_ids"]) for t in texts]       # untruncated length
fed_len = model.tokenize(texts)["attention_mask"].sum(dim=1).tolist()                      # what encode() really feeds the network
truncated = [(c["chunk_id"], f, u) for c, f, u in zip(CHUNKS, full_len, fed_len) if u < f]
mismatch = [c["chunk_id"] for c, f in zip(CHUNKS, full_len) if f != c["n_tokens"]]
print(f"chunks that would be truncated: {truncated if truncated else 'none'}")
print(f"chunks whose Stage-2 token count disagrees with the loaded model's tokenizer: {mismatch if mismatch else 'none'}")
print(f"truncation check: {len(texts) - len(truncated)}/{len(texts)} chunks are fed in full. longest chunk = {max(full_len)} tokens "
      f"({CHUNKS[int(np.argmax(full_len))]['chunk_id']}), window = {model.max_seq_length} "
      f"(headroom {model.max_seq_length - max(full_len)} tokens)")
assert not truncated and not mismatch

The `tokenize` method is deprecated, please use `preprocess` instead.


chunks that would be truncated: none
chunks whose Stage-2 token count disagrees with the loaded model's tokenizer: none
truncation check: 98/98 chunks are fed in full. longest chunk = 227 tokens (ck-031), window = 256 (headroom 29 tokens)


In [16]:
# all-MiniLM-L6-v2 needs no query/passage prefix. Vectors are L2-normalised so inner product == cosine similarity.
t0 = time.perf_counter()
E = model.encode(texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")
embed_seconds = time.perf_counter() - t0
norms = np.linalg.norm(E, axis=1)
print(f"embeddings: shape={E.shape} dtype={E.dtype}  encode time={embed_seconds:.2f}s on {EMBED_DEVICE}")
print(f"L2 norms: min={norms.min():.6f} max={norms.max():.6f} max|norm-1|={np.abs(norms - 1).max():.2e}")
assert E.shape == (len(CHUNKS), DIM) and np.isfinite(E).all()
assert np.allclose(norms, 1.0, atol=1e-5), "vectors are not unit length"
np.save(EMB_PATH, E)

embeddings: shape=(98, 384) dtype=float32  encode time=0.81s on cpu
L2 norms: min=1.000000 max=1.000000 max|norm-1|=1.19e-07


In [17]:
index = faiss.IndexFlatIP(DIM)       # exact inner-product search; with unit vectors this is cosine similarity
index.add(E)
faiss.write_index(index, str(INDEX_PATH))
print(f"FAISS IndexFlatIP: ntotal={index.ntotal}, d={index.d}, metric=inner product, "
      f"on-disk size={INDEX_PATH.stat().st_size:,} bytes (vectors alone: {E.nbytes:,} bytes)")
assert index.ntotal == len(CHUNKS) and index.d == DIM

FAISS IndexFlatIP: ntotal=98, d=384, metric=inner product, on-disk size=150,573 bytes (vectors alone: 150,528 bytes)


In [18]:
# 1) self-retrieval: every chunk's own vector must find itself first with similarity ~1.0
D, I = index.search(E, 1)
hash_of = [c["text_sha256"] for c in CHUNKS]
self_hits = sum(hash_of[I[i, 0]] == hash_of[i] for i in range(len(CHUNKS)))
print(f"[1] self-retrieval: {self_hits}/{len(CHUNKS)} chunks return themselves at rank 1; min top-1 score={D.min():.6f}")
assert self_hits == len(CHUNKS) and D.min() > 0.9999

# 2) inner product == cosine: compare FAISS scores with an explicit cosine formula on random pairs
rng = np.random.default_rng(0)
a_idx, b_idx = rng.integers(0, len(CHUNKS), 200), rng.integers(0, len(CHUNKS), 200)
cos = np.array([E[a] @ E[b] / (np.linalg.norm(E[a]) * np.linalg.norm(E[b])) for a, b in zip(a_idx, b_idx)])
ip = np.array([index.reconstruct(int(a)) @ E[b] for a, b in zip(a_idx, b_idx)])
print(f"[2] cosine vs inner product on 200 random pairs: max abs difference = {np.abs(cos - ip).max():.2e}")
assert np.allclose(cos, ip, atol=1e-5)

# 3) determinism: encoding again gives the same vectors
E2 = model.encode(texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")
print(f"[3] re-encode determinism: max abs difference = {np.abs(E - E2).max():.2e}")
assert np.allclose(E, E2, atol=1e-6)

# 4) the index written to disk behaves identically after reloading
D2, I2 = faiss.read_index(str(INDEX_PATH)).search(E, 5)
D1, I1 = index.search(E, 5)
print(f"[4] on-disk index round-trip: identical top-5 ids={np.array_equal(I1, I2)}, identical scores={np.allclose(D1, D2)}")
assert np.array_equal(I1, I2) and np.allclose(D1, D2)

# 5) query path plumbing: one natural-language query that is NOT part of the evaluation set. We check only that the
#    mechanics are valid (ids in range and unique, scores descending and within [-1, 1]); relevance is NOT judged here.
SMOKE_QUERY = "How many dice are rolled each turn?"
q = model.encode([SMOKE_QUERY], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
qs, qi = index.search(q, 10)
assert q.shape == (1, DIM) and abs(np.linalg.norm(q) - 1) < 1e-5
assert len(set(qi[0])) == 10 and qi.min() >= 0 and qi.max() < len(CHUNKS)
assert np.all(np.diff(qs[0]) <= 1e-6) and np.all(np.abs(qs[0]) <= 1.0 + 1e-6)
print(f"[5] smoke query (plumbing only): Top-10 ids/scores returned for {SMOKE_QUERY!r}")
print("    " + ", ".join(f"{CHUNKS[i]['chunk_id']}:{s:.3f}" for i, s in zip(qi[0], qs[0])))
print("    valid: 10 unique in-range ids, scores descending, all within [-1, 1]. No relevance claim is made.")

# 6) the index order equals the corpus order, so a FAISS id is a position in the frozen chunk list
print("[6] FAISS id i  <->  CHUNKS[i]: ", ", ".join(f"{i}->{CHUNKS[i]['chunk_id']}" for i in (0, 1, 49, 97)), "(ids are positions)")

[1] self-retrieval: 98/98 chunks return themselves at rank 1; min top-1 score=1.000000
[2] cosine vs inner product on 200 random pairs: max abs difference = 5.96e-08


[3] re-encode determinism: max abs difference = 0.00e+00
[4] on-disk index round-trip: identical top-5 ids=True, identical scores=True
[5] smoke query (plumbing only): Top-10 ids/scores returned for 'How many dice are rolled each turn?'
    ck-070:0.520, ck-016:0.442, ck-069:0.425, ck-018:0.423, ck-042:0.386, ck-072:0.384, ck-041:0.376, ck-029:0.373, ck-009:0.318, ck-004:0.311
    valid: 10 unique in-range ids, scores descending, all within [-1, 1]. No relevance claim is made.
[6] FAISS id i  <->  CHUNKS[i]:  0->ck-001, 1->ck-002, 49->ck-050, 97->ck-098 (ids are positions)


## Stage 3b. Flag repeated set-up text and overlapping neighbours (analysis aid, corpus unchanged)
Two sources of near-duplicate chunks can crowd a Top-3 and must be considered when reading Stage 4/5 results:
* **overlap neighbours**: by construction a continuation chunk repeats one sentence of its predecessor;
* **repeated text elsewhere**: e.g. the set-up instructions appear both on page 4 and on the page-20 game overview.

The similarity threshold is read off the distribution of chunk-to-chunk cosines, **before** any retrieval is run.

In [19]:
S = E @ E.T
np.fill_diagonal(S, -1.0)
upper = S[np.triu_indices(len(CHUNKS), 1)]
print(f"chunk-to-chunk cosine over {upper.size:,} pairs: median={np.median(upper):.2f}  p95={np.percentile(upper, 95):.2f}  "
      f"p99={np.percentile(upper, 99):.2f}  max={upper.max():.2f}")

# (a) overlap neighbours, verified against the text itself: the longest line-aligned prefix of the chunk that is also a
#     verbatim ending of its predecessor is the repeated overlap text.
def overlap_text(prev: str, cur: str) -> str:
    best = ""
    for k in [i for i, ch in enumerate(cur) if ch == "\n"] + [len(cur)]:
        if cur[:k] and prev.rstrip().endswith(cur[:k]) and len(cur[:k]) > len(best):
            best = cur[:k]
    return best


overlap_pairs, overlap_words = [], []
for i, c in enumerate(CHUNKS):
    if c["has_overlap"]:
        shared = overlap_text(CHUNKS[i - 1]["text"], c["text"])
        assert shared, f"{c['chunk_id']} is flagged as overlapping but starts with nothing its predecessor ends with"
        overlap_pairs.append((CHUNKS[i - 1]["chunk_id"], c["chunk_id"], float(S[i - 1, i])))
        overlap_words.append(len(shared.split()))
assert len(overlap_pairs) == sum(c["has_overlap"] for c in CHUNKS)
print(f"(a) overlap neighbours: {len(overlap_pairs)} pairs verified to share verbatim text "
      f"(shared length {min(overlap_words)}-{max(overlap_words)} words; cosine of the pair: "
      f"median={np.median([p[2] for p in overlap_pairs]):.2f}, max={max(p[2] for p in overlap_pairs):.2f})")

# (b) highly similar pairs that are NOT overlap neighbours
DUP_COS = 0.85
overlap_set = {(a, b) for a, b, _ in overlap_pairs}
similar = []
for i in range(len(CHUNKS)):
    for j in range(i + 1, len(CHUNKS)):
        if S[i, j] >= DUP_COS and (CHUNKS[i]["chunk_id"], CHUNKS[j]["chunk_id"]) not in overlap_set:
            same_sec = CHUNKS[i]["section"] == CHUNKS[j]["section"]
            same_page = CHUNKS[i]["page_start"] == CHUNKS[j]["page_start"]
            kind = ("same section, same page" if same_sec and same_page else
                    "same section name on another page" if same_sec else "different sections")
            similar.append(dict(a=CHUNKS[i]["chunk_id"], b=CHUNKS[j]["chunk_id"], cosine=round(float(S[i, j]), 4),
                                pages=f"p{CHUNKS[i]['page_start']} / p{CHUNKS[j]['page_start']}", kind=kind))
similar.sort(key=lambda d: -d["cosine"])
print(f"(b) non-overlap pairs with cosine >= {DUP_COS}: {len(similar)}")
for d in similar[:12]:
    print(f"    {d['a']} ~ {d['b']}  cos={d['cosine']:.3f}  {d['pages']:12s} {d['kind']}")

# (c) the set-up text repeated on page 4 and page 20, regardless of threshold
setup_a = [i for i, c in enumerate(CHUNKS) if c["page_start"] == 4 and c["section"].startswith(("Setting Up the Game", "Set-up Phase"))]
setup_b = [i for i, c in enumerate(CHUNKS) if c["page_start"] == 20]
print("(c) page-4 set-up chunks vs page-20 overview chunks (cosine):")
for i in setup_a:
    print("    " + "  ".join(f"{CHUNKS[i]['chunk_id']}~{CHUNKS[j]['chunk_id']}={S[i, j]:.3f}" for j in setup_b))

flags = dict(similarity_threshold=DUP_COS, overlap_neighbours=[dict(prev=a, chunk=b, cosine=round(c, 4)) for a, b, c in overlap_pairs],
             similar_pairs=similar,
             setup_repeat_pairs=[dict(a=CHUNKS[i]["chunk_id"], b=CHUNKS[j]["chunk_id"], cosine=round(float(S[i, j]), 4))
                                 for i in setup_a for j in setup_b])
(OUT_DIR / "near_duplicate_pairs.json").write_text(json.dumps(flags, indent=1))


def duplicate_flags(chunk_ids):
    """Stage 4/5 helper: which of these retrieved chunk ids are overlap neighbours or near-duplicates of each other?"""
    ids = set(chunk_ids)
    notes = [f"{d['prev']} -> {d['chunk']} overlap" for d in flags["overlap_neighbours"] if {d["prev"], d["chunk"]} <= ids]
    notes += [f"{d['a']} ~ {d['b']} cos={d['cosine']}" for d in flags["similar_pairs"] + flags["setup_repeat_pairs"]
              if {d["a"], d["b"]} <= ids and d["cosine"] >= DUP_COS]
    return notes

chunk-to-chunk cosine over 4,753 pairs: median=0.25  p95=0.56  p99=0.71  max=0.93
(a) overlap neighbours: 23 pairs verified to share verbatim text (shared length 6-30 words; cosine of the pair: median=0.71, max=0.90)
(b) non-overlap pairs with cosine >= 0.85: 3
    ck-010 ~ ck-023  cos=0.925  p3 / p6      different sections
    ck-013 ~ ck-097  cos=0.892  p4 / p20     same section name on another page
    ck-062 ~ ck-093  cos=0.886  p12 / p17    different sections
(c) page-4 set-up chunks vs page-20 overview chunks (cosine):
    ck-013~ck-097=0.893  ck-013~ck-098=0.543
    ck-014~ck-097=0.493  ck-014~ck-098=0.599
    ck-015~ck-097=0.585  ck-015~ck-098=0.816


In [20]:
HASH_AFTER = corpus_hash(CHUNKS)                                   # in-memory object that was embedded
HASH_AFTER_DISK = corpus_hash(load_frozen_chunks())                # re-read from disk, re-verified against the manifest
FILE_HASH_AFTER = sha256_file(DATA_DIR / "chunks.jsonl")
print(f"[after]  corpus_sha256 (memory) = {HASH_AFTER}")
print(f"[after]  corpus_sha256 (disk)   = {HASH_AFTER_DISK}")
print(f"[after]  data/chunks.jsonl file sha256 = {FILE_HASH_AFTER}")
assert HASH_BEFORE == HASH_AFTER == HASH_AFTER_DISK == FROZEN_CORPUS_SHA256
assert FILE_HASH_BEFORE == FILE_HASH_AFTER
assert [c["text_sha256"] for c in CHUNKS] == [hashlib.sha256(t.encode()).hexdigest() for t in texts], "embedded texts differ from chunks"
print("corpus verified UNCHANGED before and after embedding/indexing: memory == disk == pinned hash, and embedded texts == chunk texts")

[after]  corpus_sha256 (memory) = a3a1772d90e3a71bcacf6f1b5a70e01dc7237f4e5def64b7381eeb1877c61950
[after]  corpus_sha256 (disk)   = a3a1772d90e3a71bcacf6f1b5a70e01dc7237f4e5def64b7381eeb1877c61950
[after]  data/chunks.jsonl file sha256 = 489f7f204881460bafc493da8b29a674fdba192291595c05b1d3210f75c032e5
corpus verified UNCHANGED before and after embedding/indexing: memory == disk == pinned hash, and embedded texts == chunk texts


In [21]:
index_manifest = dict(
    embedding_model=EMBED_MODEL, device=EMBED_DEVICE, dim=DIM, max_seq_length=model.max_seq_length, normalised=True,
    index_class="faiss.IndexFlatIP", n_vectors=index.ntotal, corpus_sha256=FROZEN_CORPUS_SHA256,
    embeddings_sha256=hashlib.sha256(E.tobytes()).hexdigest(),
    note="embeddings_sha256 can differ across CPUs/BLAS builds in the last bits; corpus_sha256 is the binding reference.",
    versions=VERSIONS)
(OUT_DIR / "index_manifest.json").write_text(json.dumps(index_manifest, indent=1))
print("wrote results/index_manifest.json and results/near_duplicate_pairs.json (no rulebook text in either)")
print("Stage 3 complete. Nothing has been retrieved or re-ranked yet.")

wrote results/index_manifest.json and results/near_duplicate_pairs.json (no rulebook text in either)
Stage 3 complete. Nothing has been retrieved or re-ranked yet.


---
# Stage 4. Baseline retrieval (Pipeline 1, retrieval half)
Scope: embed a query with the Bi-Encoder, retrieve the **Top 10** from the frozen FAISS index (cosine = inner product on unit
vectors) and keep the first **3** as the baseline context. **No Cross-Encoder, no LLM, no timing in this stage.**

**Rules fixed before any result was looked at**
* `K_INIT = 10`, `K_FINAL = 3` (the assignment).
* Outcome of a query, given its hand-labelled gold chunk(s) (Stage 2e):
  * **success**: at least one gold chunk is in the Top 3 (the baseline already supplies the evidence);
  * **retrieval stumble**: the best gold chunk is at rank 4-10 (evidence is in the pool but not in the context; a re-ranker *could* promote it);
  * **candidate-pool miss**: no gold chunk in the Top 10 (a re-ranker cannot help, because it only reorders these 10).
* Nothing is removed, re-ordered or re-labelled after seeing the results. Duplicate / overlap chunks are only **flagged**.

In [22]:
K_INIT, K_FINAL = 10, 3

CHUNKS = load_frozen_chunks()                                  # also re-checks the hash stored in the corpus manifest
index_manifest = json.load(open(OUT_DIR / "index_manifest.json"))
E_disk = np.load(EMB_PATH)
index = faiss.read_index(str(INDEX_PATH))
V = index.reconstruct_n(0, index.ntotal)

checks = {
    "corpus hash == pinned FROZEN_CORPUS_SHA256": corpus_hash(CHUNKS) == FROZEN_CORPUS_SHA256,
    "index manifest was built from the same corpus": index_manifest["corpus_sha256"] == FROZEN_CORPUS_SHA256,
    "embeddings.npy hash == index manifest": hashlib.sha256(E_disk.tobytes()).hexdigest() == index_manifest["embeddings_sha256"],
    "index holds exactly one vector per chunk": index.ntotal == len(CHUNKS) == 98,
    "index dimension == embedding dimension": index.d == E_disk.shape[1] == DIM,
    "vectors stored in the index == embeddings.npy": bool(np.allclose(V, E_disk, atol=1e-7)),
    "index metric is inner product": index.metric_type == faiss.METRIC_INNER_PRODUCT,
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "frozen artefacts are inconsistent; do not continue"
print(f"corpus_sha256      = {corpus_hash(CHUNKS)}")
print(f"index file sha256  = {sha256_file(INDEX_PATH)}")
print(f"chunks.jsonl sha256= {sha256_file(DATA_DIR / 'chunks.jsonl')}")

PASS  corpus hash == pinned FROZEN_CORPUS_SHA256
PASS  index manifest was built from the same corpus
PASS  embeddings.npy hash == index manifest
PASS  index holds exactly one vector per chunk
PASS  index dimension == embedding dimension
PASS  vectors stored in the index == embeddings.npy
PASS  index metric is inner product
corpus_sha256      = a3a1772d90e3a71bcacf6f1b5a70e01dc7237f4e5def64b7381eeb1877c61950
index file sha256  = 2afbdb3d589671e561511d25ccf2d15d46ed827fc77b904d924e1ab8168ff733
chunks.jsonl sha256= 489f7f204881460bafc493da8b29a674fdba192291595c05b1d3210f75c032e5


In [23]:
DUPS = json.load(open(OUT_DIR / "near_duplicate_pairs.json"))        # produced in Stage 3b; used for flagging only


def baseline_retrieve(query: str, k: int = K_INIT):
    """Bi-Encoder retrieval: embed the query (unit length) and return the Top-k chunks with 1-based rank and cosine score."""
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")
    scores, ids = index.search(q, k)
    rows = []
    for rank, (i, s) in enumerate(zip(ids[0], scores[0]), 1):
        c = CHUNKS[int(i)]
        rows.append(dict(rank=rank, faiss_id=int(i), chunk_id=c["chunk_id"], score=float(s),
                         page=c["page_start"], section=c["section"], n_tokens=c["n_tokens"]))
    return rows


def baseline_context(rows, k: int = K_FINAL):
    """The baseline passes the first k retrieved chunks, in retrieval order, to the LLM."""
    return rows[:k]


def check_ranking(rows, k):
    assert [r["rank"] for r in rows] == list(range(1, k + 1))
    assert len({r["chunk_id"] for r in rows}) == k
    assert all(rows[i]["score"] >= rows[i + 1]["score"] for i in range(k - 1))
    assert all(-1.0 - 1e-6 <= r["score"] <= 1.0 + 1e-6 for r in rows)


def duplicate_notes(ids):
    """Overlap neighbours / near-duplicates (cosine >= 0.85) that are BOTH inside `ids`. Flag only, never remove."""
    ids = set(ids)
    notes = [("overlap", d["prev"], d["chunk"], d["cosine"]) for d in DUPS["overlap_neighbours"] if {d["prev"], d["chunk"]} <= ids]
    notes += [("near-duplicate", d["a"], d["b"], d["cosine"]) for d in DUPS["similar_pairs"] + DUPS["setup_repeat_pairs"]
              if {d["a"], d["b"]} <= ids and d["cosine"] >= DUPS["similarity_threshold"]]
    return notes


def classify_outcome(gold_ids, top10_ids):
    in_pool = {g: top10_ids.index(g) + 1 for g in gold_ids if g in top10_ids}
    best = min(in_pool.values()) if in_pool else None
    if best is None:
        return "candidate-pool miss", None
    return ("success" if best <= K_FINAL else "retrieval stumble"), best


def full_rank_of(query, chunk_ids):
    """Information only: where the chunks sit in the complete 98-chunk ranking (not used by any pipeline)."""
    allrows = baseline_retrieve(query, len(CHUNKS))
    pos = {r["chunk_id"]: (r["rank"], r["score"]) for r in allrows}
    return {cid: pos[cid] for cid in chunk_ids}


print(f"baseline retriever ready: Top-{K_INIT} retrieved, first {K_FINAL} passed on as context")

baseline retriever ready: Top-10 retrieved, first 3 passed on as context


In [24]:
BASELINE_RESULTS = {}
for name, qd in CANDIDATES.items():
    rows = baseline_retrieve(qd["question"], K_INIT)
    check_ranking(rows, K_INIT)
    ids10 = [r["chunk_id"] for r in rows]
    gold = qd["gold_chunk_ids"]
    outcome, best_rank = classify_outcome(gold, ids10)
    dup10 = duplicate_notes(ids10)
    dup3 = duplicate_notes(ids10[:K_FINAL])
    BASELINE_RESULTS[name] = dict(question=qd["question"], gold_chunk_ids=gold, outcome=outcome, best_gold_rank=best_rank,
                                  top10=rows, top3_chunk_ids=ids10[:K_FINAL], duplicate_notes_top10=dup10,
                                  duplicate_notes_top3=dup3, gold_full_ranking=full_rank_of(qd["question"], gold))

    print("=" * 112)
    print(f"{name}\n  question: {qd['question']}\n  gold chunk(s): {gold}")
    print(f"  {'rank':>4}  {'chunk':6} {'score':>7}  {'page':>4}  {'tok':>4}  {'section':44} flags")
    nbr = {x for n in dup10 for x in n[1:3]}
    for r in rows:
        flags_ = []
        if r["chunk_id"] in gold:
            flags_.append("GOLD")
        if r["rank"] <= K_FINAL:
            flags_.append("-> LLM context")
        if r["chunk_id"] in nbr:
            flags_.append("dup/overlap")
        print(f"  {r['rank']:>4}  {r['chunk_id']:6} {r['score']:7.4f}  {r['page']:>4}  {r['n_tokens']:>4}  {r['section'][:44]:44} {' | '.join(flags_)}")
    print(f"  OUTCOME: {outcome}" + (f" (best gold chunk at rank {best_rank})" if best_rank else " (no gold chunk in the Top 10)"))
    for g, (rk, sc) in BASELINE_RESULTS[name]["gold_full_ranking"].items():
        print(f"    gold {g}: rank {rk} of {len(CHUNKS)} in the full ranking, score {sc:.4f}  (rank-1 score {rows[0]['score']:.4f}, gap {rows[0]['score'] - sc:.4f})")
    print("  duplicate/overlap warnings inside Top 10:", [f"{k}: {a}~{b} cos={c}" for k, a, b, c in dup10] or "none")
    print("  duplicate/overlap warnings inside Top 3 (the LLM context):", [f"{k}: {a}~{b} cos={c}" for k, a, b, c in dup3] or "none")

Q1 robber before the first barbarian attack
  question: If I roll a 7 before the barbarians have attacked for the first time, can I move the robber?
  gold chunk(s): ['ck-021']
  rank  chunk    score  page   tok  section                                      flags
     1  ck-021  0.6287     5   180  Specific Actions > Dice Rolls                GOLD | -> LLM context
     2  ck-069  0.5674    13   117  Easy Start Variant                           -> LLM context | dup/overlap
     3  ck-070  0.5274    13   118  Easy Start Variant                           -> LLM context | dup/overlap
     4  ck-082  0.5025    16   107  The Almanac > Bishop (2)                     
     5  ck-061  0.4482    12    97  Attack of the Barbarians! > Losing Your Last 
     6  ck-024  0.4402     6    39  Specific Actions > City Walls                
     7  ck-066  0.4317    13   180  Seafarers of Catan Variant                   
     8  ck-018  0.4174     5   154  Specific Actions > Dice Rolls                
   

In [25]:
baseline_file = dict(
    stage=4, pipeline="baseline (Bi-Encoder only)", embed_model=EMBED_MODEL, k_init=K_INIT, k_final=K_FINAL,
    corpus_sha256=FROZEN_CORPUS_SHA256, index_file_sha256=sha256_file(INDEX_PATH),
    outcome_rules={"success": "a gold chunk is in the Top 3", "retrieval stumble": "best gold chunk at rank 4-10",
                   "candidate-pool miss": "no gold chunk in the Top 10"},
    queries=BASELINE_RESULTS)
(OUT_DIR / "baseline_top10.json").write_text(json.dumps(baseline_file, indent=1))
assert corpus_hash(load_frozen_chunks()) == FROZEN_CORPUS_SHA256, "corpus changed during Stage 4"
print("saved results/baseline_top10.json | corpus hash re-verified after retrieval: unchanged")
print("summary:", {n.split()[0]: (r["outcome"], r["best_gold_rank"]) for n, r in BASELINE_RESULTS.items()})
print("Stage 4 complete. No Cross-Encoder, generation or timing has been run.")

saved results/baseline_top10.json | corpus hash re-verified after retrieval: unchanged
summary: {'Q1': ('success', 1), 'Q2': ('success', 1), 'Q3': ('success', 2)}
Stage 4 complete. No Cross-Encoder, generation or timing has been run.


---
# Stage 4b. Pre-registered failure-query search (baseline retrieval only)
Stage 4 showed that the three hand-picked questions are all easy for the Bi-Encoder (they reuse the rulebook's own words).
The assignment needs questions where the baseline *stumbles*. To avoid cherry-picking, the questions below were **written
after reading the corpus and before any of them was run through the index**; their gold evidence phrases were resolved to
chunks by string match only. The list is hashed, the hash is pinned in this notebook, and **every** query is reported (k of N),
including the ones the baseline answers correctly.

* 16 questions = 4 each of: keyword trap (synonyms / lexical distractor), exception located elsewhere,
  negation or role constraint, and other buried detail.
* Same outcome rules as Stage 4 (success / retrieval stumble / candidate-pool miss). Any gold chunk in `evidence` counts.
* `likely_distractors` are my *predictions* of competing chunks, recorded in advance so they can be checked against the results.
* Disclosure: I had already seen the Stage 4 results for Q1-Q3 when I wrote these, but none of the 16 was run before the hash was fixed.

In [26]:
PREREG = [
 # ---- A. Keyword trap / distractor: wording that overlaps a different passage; the answer uses other terms ----------------
 dict(id="A1", category="keyword trap",
      question="How many points is a city worth if one of the yellow gates is sitting on it?",
      evidence=[["worth a total of 4 victory points"]], likely_distractors=["ck-019", "ck-020", "ck-037"]),
 dict(id="A2", category="keyword trap",
      question="None of my hexes paid out on my roll. Is there anything that still lets me take something?",
      evidence=[["do not receive any resources or commodities"]], likely_distractors=["ck-018", "ck-020", "ck-021"]),
 dict(id="A3", category="keyword trap",
      question="Do I get anything extra while I'm holding the little wooden cone?",
      evidence=[["you receive 1 victory point"], ["receive 1 victory point for as long as you control the merchant"]],
      likely_distractors=["ck-013", "ck-097", "ck-004"]),
 dict(id="A4", category="keyword trap",
      question="Do stronger soldiers need more grain to get ready for action?",
      evidence=[["does not affect the cost to activate it"]], likely_distractors=["ck-028", "ck-056", "ck-047"]),
 # ---- E. Exceptions and corner cases: a general rule is easy to find, the exception sits in another chunk ------------------
 dict(id="E1", category="exception",
      question="Can I ever get a mighty knight onto the board without having built the Fortress?",
      evidence=[["mighty", "even if you have not yet built"]], likely_distractors=["ck-028", "ck-036", "ck-081"]),
 dict(id="E2", category="exception",
      question="Do I need a knight of my own to displace an opponent's knight?",
      evidence=[["without using a knight of your own"]], likely_distractors=["ck-050", "ck-051"]),
 dict(id="E3", category="exception",
      question="What if several players tie for the biggest knight army after we beat the barbarians?",
      evidence=[["no player is declared defender of catan"]], likely_distractors=["ck-057", "ck-058"]),
 dict(id="E4", category="exception",
      question="Which production numbers am I not allowed to swap when I move number tokens around?",
      evidence=[["but not 2, 12, 6, or 8"]], likely_distractors=["ck-097", "ck-013"]),
 # ---- N. Negation and constraints: "may not", "only", role limits --------------------------------------------------------
 dict(id="N1", category="negation/constraint",
      question="Can I use two cards that cut the price of one city improvement on the same upgrade?",
      evidence=[["not combine 2 crane cards"]], likely_distractors=["ck-077", "ck-080"]),
 dict(id="N2", category="negation/constraint",
      question="Can I raid the hand of a player who is behind me on points?",
      evidence=[["only play this card on an opponent who has more victory points than you"]],
      likely_distractors=["ck-090", "ck-088", "ck-087"]),
 dict(id="N3", category="negation/constraint",
      question="Do all of the players have to agree before one of the optional rules is used?",
      evidence=[["strictly optional and should only be used if agreed on by all players"]],
      likely_distractors=["ck-065", "ck-066", "ck-069"]),
 dict(id="N4", category="negation/constraint",
      question="Can the robber stop me from using the merchant's cheaper trades?",
      evidence=[["the robber does not affect the merchant"]], likely_distractors=["ck-062", "ck-091"]),
 # ---- O. Other: the vital detail is buried in a long, list-like or very short chunk -------------------------------------
 dict(id="O1", category="other (buried detail)",
      question="With two cities that each have a wall, how many cards can I keep when a seven comes up?",
      evidence=[["safely hold up to 11 cards"]], likely_distractors=["ck-010", "ck-074", "ck-021"]),
 dict(id="O2", category="other (buried detail)",
      question="How many ship faces are on the event die?",
      evidence=[["1 event die (with 3 ships and 3 city gate symbols)"]], likely_distractors=["ck-018", "ck-019", "ck-054"]),
 dict(id="O3", category="other (buried detail)",
      question="How strong are the barbarians when they land?",
      evidence=[["strength of the barbarians is equal to the number of cities"]], likely_distractors=["ck-056", "ck-054", "ck-058"]),
 dict(id="O4", category="other (buried detail)",
      question="Can a settlement next to a pasture hex give me cloth?",
      evidence=[["commodities are produced only by cities"]], likely_distractors=["ck-003", "ck-020", "ck-031"]),
]

PREREG_SHA256 = "312f1b2ddaff694c4ed8600fb1c77b9ac000b791860c3d6ef38202bc6c095476"   # fixed on 2026-10-18 00:00 IST, before any run


def resolve_gold(evidence):
    gold = set()
    for ev in evidence:
        hits = [c["chunk_id"] for c in find_chunks(*ev)]
        assert hits, f"evidence phrase not found in the frozen corpus: {ev}"
        gold |= set(hits)
    return sorted(gold)


PREREG_Q = [dict(q, gold_chunk_ids=resolve_gold(q["evidence"])) for q in PREREG]
canon = json.dumps(dict(corpus_sha256=FROZEN_CORPUS_SHA256, queries=PREREG_Q), sort_keys=True, ensure_ascii=False)
prereg_hash = hashlib.sha256(canon.encode()).hexdigest()
saved = json.load(open(OUT_DIR / "stage4b_preregistration.json"))
print(f"questions: {len(PREREG_Q)}  by category: {dict(collections.Counter(q['category'] for q in PREREG_Q))}")
print(f"pre-registration hash (recomputed here) = {prereg_hash}")
print(f"pre-registration hash pinned in code    = {PREREG_SHA256}")
print(f"hash stored in results/stage4b_preregistration.json = {saved['prereg_sha256']}")
assert prereg_hash == PREREG_SHA256 == saved["prereg_sha256"], "the question set or its gold labels changed after pre-registration"
print("PASS: questions and gold labels are exactly the ones registered before any retrieval was run")

questions: 16  by category: {'keyword trap': 4, 'exception': 4, 'negation/constraint': 4, 'other (buried detail)': 4}
pre-registration hash (recomputed here) = 312f1b2ddaff694c4ed8600fb1c77b9ac000b791860c3d6ef38202bc6c095476
pre-registration hash pinned in code    = 312f1b2ddaff694c4ed8600fb1c77b9ac000b791860c3d6ef38202bc6c095476
hash stored in results/stage4b_preregistration.json = 312f1b2ddaff694c4ed8600fb1c77b9ac000b791860c3d6ef38202bc6c095476
PASS: questions and gold labels are exactly the ones registered before any retrieval was run


In [27]:
S4B = []
for q in PREREG_Q:
    rows = baseline_retrieve(q["question"], K_INIT)
    check_ranking(rows, K_INIT)
    ids10 = [r["chunk_id"] for r in rows]
    outcome, best = classify_outcome(q["gold_chunk_ids"], ids10)
    full = full_rank_of(q["question"], q["gold_chunk_ids"])
    rec = dict(id=q["id"], category=q["category"], question=q["question"], gold_chunk_ids=q["gold_chunk_ids"],
               outcome=outcome, best_gold_rank=best, top10=rows, top3_chunk_ids=ids10[:K_FINAL],
               gold_full_ranking={g: dict(rank=r, score=s) for g, (r, s) in full.items()},
               distractors_predicted=q["likely_distractors"],
               distractors_in_top3=[d for d in q["likely_distractors"] if d in ids10[:K_FINAL]],
               distractors_in_top10=[d for d in q["likely_distractors"] if d in ids10],
               duplicate_notes_top3=duplicate_notes(ids10[:K_FINAL]), duplicate_notes_top10=duplicate_notes(ids10))
    S4B.append(rec)

    print("=" * 108)
    print(f"{q['id']} [{q['category']}]  {q['question']}")
    print(f"   gold: {q['gold_chunk_ids']}   predicted distractors: {q['likely_distractors']}")
    nbr = {x for n in rec["duplicate_notes_top10"] for x in n[1:3]}
    for r in rows:
        tags = []
        if r["chunk_id"] in q["gold_chunk_ids"]:
            tags.append("GOLD")
        if r["chunk_id"] in q["likely_distractors"]:
            tags.append("predicted-distractor")
        if r["chunk_id"] in nbr:
            tags.append("dup/overlap")
        ctx = "->LLM" if r["rank"] <= K_FINAL else "     "
        print(f"   {r['rank']:>2} {ctx} {r['chunk_id']} {r['score']:.4f} p{r['page']:<2} {r['section'][:42]:42} {' | '.join(tags)}")
    g = ", ".join(f"{k}: rank {v['rank']}/{len(CHUNKS)}" for k, v in rec["gold_full_ranking"].items())
    print(f"   OUTCOME: {outcome.upper()}  (best gold in Top-10 at rank {best}; full ranking: {g})" if best else
          f"   OUTCOME: {outcome.upper()}  (full ranking: {g})")
    if rec["duplicate_notes_top3"]:
        print("   duplicate/overlap inside the LLM context (Top 3):", [f"{a}~{b}" for _, a, b, _ in rec["duplicate_notes_top3"]])

A1 [keyword trap]  How many points is a city worth if one of the yellow gates is sitting on it?
   gold: ['ck-038']   predicted distractors: ['ck-019', 'ck-020', 'ck-037']
    1 ->LLM ck-019 0.5174 p5  Specific Actions > Dice Rolls              predicted-distractor | dup/overlap
    2 ->LLM ck-039 0.4791 p8  The Metropolis                             
    3 ->LLM ck-037 0.4770 p8  The Metropolis                             predicted-distractor
    4       ck-020 0.4503 p5  Specific Actions > Dice Rolls              predicted-distractor | dup/overlap
    5       ck-023 0.4459 p6  Specific Actions > City Walls              dup/overlap
    6       ck-010 0.4404 p3  New Game Pieces > City Walls               dup/overlap
    7       ck-098 0.4197 p20 Setting Up the Game                        
    8       ck-041 0.4066 p9  Progress Cards > Acquiring Progress Cards  
    9       ck-016 0.3956 p5  Turn Overview                              
   10       ck-004 0.3948 p2  Game Components       

In [28]:
def tally(records):
    return dict(collections.Counter(r["outcome"] for r in records))


print("=" * 108)
print(f"STAGE 4b RESULT: N = {len(S4B)} pre-registered questions   ->  {tally(S4B)}")
for cat in dict.fromkeys(r["category"] for r in S4B):
    sub = [r for r in S4B if r["category"] == cat]
    print(f"   {cat:24s} {tally(sub)}")
stumbles = [r for r in S4B if r["outcome"] == "retrieval stumble"]
misses = [r for r in S4B if r["outcome"] == "candidate-pool miss"]
print(f"retrieval stumbles (gold at rank 4-10, a re-ranker could promote it): {len(stumbles)} of {len(S4B)} -> "
      f"{[(r['id'], r['best_gold_rank']) for r in stumbles] or 'none'}")
print(f"candidate-pool misses (gold outside the Top 10, a re-ranker cannot help): {len(misses)} of {len(S4B)} -> "
      f"{[(r['id'], min(v['rank'] for v in r['gold_full_ranking'].values())) for r in misses] or 'none'}")
earlier = [dict(outcome=v["outcome"]) for v in BASELINE_RESULTS.values()]
print(f"all baseline queries run so far (Stage 4 Q1-Q3 + Stage 4b): N = {len(S4B) + len(earlier)} -> {tally(S4B + earlier)}")
hyp = sum(bool(r["distractors_in_top3"]) for r in S4B)
print(f"prediction check: at least one predicted distractor was inside the Top 3 for {hyp} of {len(S4B)} questions")

print(f"{'id':3} {'category':24} {'outcome':20} best-gold-rank")
for r in S4B:
    print(f"{r['id']:3} {r['category']:24} {r['outcome']:20} {r['best_gold_rank']}")

(OUT_DIR / "stage4b_baseline.json").write_text(json.dumps(dict(
    stage="4b", pipeline="baseline (Bi-Encoder only)", prereg_sha256=PREREG_SHA256, corpus_sha256=FROZEN_CORPUS_SHA256,
    index_file_sha256=sha256_file(INDEX_PATH), k_init=K_INIT, k_final=K_FINAL, n_questions=len(S4B), tally=tally(S4B),
    queries=S4B), indent=1))
assert corpus_hash(load_frozen_chunks()) == FROZEN_CORPUS_SHA256, "corpus changed during Stage 4b"
print("saved results/stage4b_baseline.json | corpus hash re-verified: unchanged | no Cross-Encoder, generation or timing was run")

STAGE 4b RESULT: N = 16 pre-registered questions   ->  {'candidate-pool miss': 4, 'success': 11, 'retrieval stumble': 1}
   keyword trap             {'candidate-pool miss': 4}
   exception                {'success': 4}
   negation/constraint      {'success': 4}
   other (buried detail)    {'success': 3, 'retrieval stumble': 1}
retrieval stumbles (gold at rank 4-10, a re-ranker could promote it): 1 of 16 -> [('O4', 6)]
candidate-pool misses (gold outside the Top 10, a re-ranker cannot help): 4 of 16 -> [('A1', 11), ('A2', 33), ('A3', 17), ('A4', 16)]
all baseline queries run so far (Stage 4 Q1-Q3 + Stage 4b): N = 19 -> {'candidate-pool miss': 4, 'success': 14, 'retrieval stumble': 1}
prediction check: at least one predicted distractor was inside the Top 3 for 9 of 16 questions
id  category                 outcome              best-gold-rank
A1  keyword trap             candidate-pool miss  None
A2  keyword trap             candidate-pool miss  None
A3  keyword trap             candidate

---
# Stage 5. Cross-Encoder re-ranking (Pipeline 2, retrieval half)
Scope: take **the same Top-10 candidates** the baseline returned for each query, score every (query, chunk) pair jointly with the
Cross-Encoder, sort by that score and keep the first 3. **No LLM answer, no timing in this stage.**

**Rules fixed before looking at any result**
* Candidate pool = the baseline Top 10, passed in as the *same list*; the re-ranker may reorder it but cannot add or drop a chunk
  (asserted for every query). Re-ranking is the only thing that differs from Pipeline 1.
* Scores are the model's raw logits (no sigmoid): unbounded, comparable only *within* one query. Ties keep the baseline order.
* Movement of a query: **promoted** = baseline stumble -> gold in the re-ranked Top 3; **demoted** = baseline success -> gold no longer
  in the Top 3; **kept** = success -> success; **still outside** = stumble -> stumble; **pool miss** = gold was never in the Top 10.
* Run on **every** query so far (Q1-Q3 and the 16 pre-registered Stage 4b questions); nothing is dropped after seeing results.

In [29]:
CE_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
ce = CrossEncoder(CE_NAME, device="cpu")
CE_MAX_LEN = ce.max_length
print(f"loaded {CE_NAME} on cpu | params={sum(p.numel() for p in ce.model.parameters()) / 1e6:.1f}M | max_length={CE_MAX_LEN} | "
      f"num_labels={ce.num_labels} | default activation={ce.activation_fn}")
assert CE_MAX_LEN == 512 and ce.num_labels == 1
CHUNK_BY_ID = {c["chunk_id"]: c for c in CHUNKS}

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10333.71it/s]

loaded cross-encoder/ms-marco-MiniLM-L-6-v2 on cpu | params=22.7M | max_length=512 | num_labels=1 | default activation=Identity()


/var/folders/nc/mwdjtcy57ss8w8gfjngyvjzh0000gn/T/ipykernel_33106/2183470040.py:3: DeprecationWarning: The `max_length` property was renamed and is now deprecated. Please use `max_seq_length` instead.
  CE_MAX_LEN = ce.max_length


In [30]:
def cross_encoder_scores(query, chunk_ids):
    """Joint (query, chunk) scoring: query and chunk go through the network together. Raw logits, one per pair."""
    pairs = [(query, CHUNK_BY_ID[cid]["text"]) for cid in chunk_ids]
    return [float(s) for s in ce.predict(pairs, batch_size=K_INIT, show_progress_bar=False, activation_fn=torch.nn.Identity())]


def rerank_top10(query, baseline_rows):
    """Re-sort EXACTLY the baseline candidates by Cross-Encoder score (descending, stable)."""
    ids = [r["chunk_id"] for r in baseline_rows]
    assert len(ids) == K_INIT and len(set(ids)) == K_INIT
    scores = cross_encoder_scores(query, ids)
    order = sorted(range(K_INIT), key=lambda i: (-scores[i], i))
    out = []
    for new_rank, i in enumerate(order, 1):
        r = baseline_rows[i]
        out.append(dict(rank=new_rank, orig_rank=r["rank"], chunk_id=r["chunk_id"], ce_score=scores[i], bi_score=r["score"],
                        page=r["page"], section=r["section"], n_tokens=r["n_tokens"]))
    assert sorted(x["chunk_id"] for x in out) == sorted(ids), "re-ranking changed the candidate pool"
    return out


def movement(base_outcome, new_outcome):
    if base_outcome == "candidate-pool miss":
        return "pool miss (re-ranker cannot help)"
    if base_outcome == "retrieval stumble":
        return "PROMOTED into Top 3" if new_outcome == "success" else "still outside Top 3"
    return "kept in Top 3" if new_outcome == "success" else "DEMOTED out of Top 3"

In [31]:
ALL_Q = [dict(id=n.split()[0], category="candidate (Stage 2e/4)", question=r["question"], gold=r["gold_chunk_ids"], base_rows=r["top10"])
         for n, r in BASELINE_RESULTS.items()]
ALL_Q += [dict(id=r["id"], category=r["category"], question=r["question"], gold=r["gold_chunk_ids"], base_rows=r["top10"]) for r in S4B]

# the pool handed to the re-ranker must be the saved baseline pool, and retrieval must be reproducible
saved4 = json.load(open(OUT_DIR / "baseline_top10.json"))["queries"]
saved4b = {q["id"]: q for q in json.load(open(OUT_DIR / "stage4b_baseline.json"))["queries"]}
for q in ALL_Q:
    ids = [r["chunk_id"] for r in q["base_rows"]]
    saved_ids = ([r["chunk_id"] for r in next(v for k, v in saved4.items() if k.startswith(q["id"]))["top10"]]
                 if q["id"].startswith("Q") else [r["chunk_id"] for r in saved4b[q["id"]]["top10"]])
    fresh = [r["chunk_id"] for r in baseline_retrieve(q["question"], K_INIT)]
    assert ids == saved_ids == fresh, f"{q['id']}: candidate pool differs from the saved baseline"
print(f"PASS: {len(ALL_Q)} queries; each candidate pool == saved baseline Top-10 == a fresh baseline retrieval")

pair_lens = [len(ce.tokenizer(q["question"], CHUNK_BY_ID[r["chunk_id"]]["text"], truncation=False)["input_ids"])
             for q in ALL_Q for r in q["base_rows"]]
too_long = [n for n in pair_lens if n > CE_MAX_LEN]
print(f"Cross-Encoder input check: {len(pair_lens)} (query, chunk) pairs, longest = {max(pair_lens)} tokens, window = {CE_MAX_LEN}, "
      f"pairs that would be truncated: {len(too_long)}")
assert not too_long

PASS: 19 queries; each candidate pool == saved baseline Top-10 == a fresh baseline retrieval
Cross-Encoder input check: 190 (query, chunk) pairs, longest = 245 tokens, window = 512, pairs that would be truncated: 0


In [32]:
S5 = []
for q in ALL_Q:
    rr = rerank_top10(q["question"], q["base_rows"])
    base_ids = [r["chunk_id"] for r in q["base_rows"]]
    new_ids = [r["chunk_id"] for r in rr]
    b_out, b_best = classify_outcome(q["gold"], base_ids)
    n_out, n_best = classify_outcome(q["gold"], new_ids)
    mv = movement(b_out, n_out)
    rec = dict(id=q["id"], category=q["category"], question=q["question"], gold_chunk_ids=q["gold"],
               baseline_outcome=b_out, baseline_best_gold_rank=b_best, rerank_outcome=n_out, rerank_best_gold_rank=n_best, movement=mv,
               baseline_top3=base_ids[:K_FINAL], rerank_top3=new_ids[:K_FINAL],
               entered_top3=[c for c in new_ids[:K_FINAL] if c not in base_ids[:K_FINAL]],
               left_top3=[c for c in base_ids[:K_FINAL] if c not in new_ids[:K_FINAL]],
               reranked_top10=rr, duplicate_notes_rerank_top3=duplicate_notes(new_ids[:K_FINAL]))
    S5.append(rec)

    print("=" * 112)
    print(f"{q['id']} [{q['category']}]  {q['question']}")
    print(f"   gold: {q['gold']}   baseline: {b_out}" + (f" (gold rank {b_best})" if b_best else "") +
          f"   ->   re-ranked: {n_out}" + (f" (gold rank {n_best})" if n_best else ""))
    print(f"   {'new':>3} {'was':>3}  {'chunk':6} {'CE logit':>9} {'bi-enc':>7}  {'section':40} flags")
    for r in rr:
        tags = []
        if r["chunk_id"] in q["gold"]:
            tags.append("GOLD")
        if r["rank"] <= K_FINAL:
            tags.append("->LLM")
        shift = r["orig_rank"] - r["rank"]
        arrow = f"{'+' if shift > 0 else ''}{shift}" if shift else "="
        print(f"   {r['rank']:>3} {r['orig_rank']:>3}  {r['chunk_id']:6} {r['ce_score']:9.3f} {r['bi_score']:7.4f}  {r['section'][:40]:40} {arrow:>3} {' | '.join(tags)}")
    print(f"   MOVEMENT: {mv}" + (f"   entered Top 3: {rec['entered_top3']}  left Top 3: {rec['left_top3']}" if rec["entered_top3"] else "   (Top-3 set unchanged)"))
    if rec["duplicate_notes_rerank_top3"]:
        print("   duplicate/overlap inside the re-ranked Top 3:", [f"{a}~{b}" for _, a, b, _ in rec["duplicate_notes_rerank_top3"]])

Q1 [candidate (Stage 2e/4)]  If I roll a 7 before the barbarians have attacked for the first time, can I move the robber?
   gold: ['ck-021']   baseline: success (gold rank 1)   ->   re-ranked: success (gold rank 1)
   new was  chunk   CE logit  bi-enc  section                                  flags
     1   1  ck-021     8.393  0.6287  Specific Actions > Dice Rolls              = GOLD | ->LLM
     2   3  ck-070     3.397  0.5274  Easy Start Variant                        +1 ->LLM
     3   2  ck-069     1.912  0.5674  Easy Start Variant                        -1 ->LLM
     4   7  ck-066    -0.364  0.4317  Seafarers of Catan Variant                +3 
     5  10  ck-068    -1.249  0.3943  Knight-Errant Variant                     +5 
     6   4  ck-082    -1.978  0.5025  The Almanac > Bishop (2)                  -2 
     7   8  ck-018    -2.245  0.4174  Specific Actions > Dice Rolls             +1 
     8   5  ck-061    -3.237  0.4482  Attack of the Barbarians! > Losing Your   -3 
     

A1 [keyword trap]  How many points is a city worth if one of the yellow gates is sitting on it?
   gold: ['ck-038']   baseline: candidate-pool miss   ->   re-ranked: candidate-pool miss
   new was  chunk   CE logit  bi-enc  section                                  flags
     1   3  ck-037    -0.614  0.4770  The Metropolis                            +2 ->LLM
     2   1  ck-019    -2.880  0.5174  Specific Actions > Dice Rolls             -1 ->LLM
     3   4  ck-020    -4.490  0.4503  Specific Actions > Dice Rolls             +1 ->LLM
     4  10  ck-004    -6.347  0.3948  Game Components                           +6 
     5   2  ck-039    -7.822  0.4791  The Metropolis                            -3 
     6   8  ck-041    -8.472  0.4066  Progress Cards > Acquiring Progress Card  +2 
     7   9  ck-016    -9.696  0.3956  Turn Overview                             +2 
     8   6  ck-010   -10.196  0.4404  New Game Pieces > City Walls              -2 
     9   5  ck-023   -10.333  0.4459  Spec

A3 [keyword trap]  Do I get anything extra while I'm holding the little wooden cone?
   gold: ['ck-062', 'ck-093']   baseline: candidate-pool miss   ->   re-ranked: candidate-pool miss
   new was  chunk   CE logit  bi-enc  section                                  flags
     1   9  ck-043   -11.168  0.1407  Progress Cards > Playing Progress Cards   +8 ->LLM
     2   2  ck-031   -11.306  0.1644  Commodities                                = ->LLM
     3  10  ck-091   -11.313  0.1366  The Almanac > Commercial Harbor (2)       +7 ->LLM
     4   8  ck-003   -11.361  0.1414  Game Components                           +4 
     5   3  ck-083   -11.364  0.1563  The Almanac > Constitution (1)            -2 
     6   1  ck-026   -11.365  0.1827  Specific Actions > Knights                -5 
     7   7  ck-030   -11.366  0.1431  Commodities                                = 
     8   4  ck-022   -11.373  0.1539  Specific Actions > Trading and Building   -4 
     9   6  ck-088   -11.383  0.1511  The A

E2 [exception]  Do I need a knight of my own to displace an opponent's knight?
   gold: ['ck-086']   baseline: success (gold rank 1)   ->   re-ranked: success (gold rank 1)
   new was  chunk   CE logit  bi-enc  section                                  flags
     1   1  ck-086     9.205  0.7184  The Almanac > Intrigue (2)                 = GOLD | ->LLM
     2   2  ck-050     7.890  0.7060  Knights > Displacing a Knight              = ->LLM
     3   3  ck-051     7.741  0.7030  Knights > Displacing a Knight              = ->LLM
     4   5  ck-084     6.002  0.5832  The Almanac > Deserter (2)                +1 
     5   7  ck-068    -1.565  0.5327  Knight-Errant Variant                     +2 
     6   9  ck-053    -2.429  0.5206  Knights > Chasing Away the Robber         +3 
     7   6  ck-027    -3.278  0.5554  Specific Actions > Activating a Knight    -1 
     8  10  ck-028    -3.612  0.5196  Specific Actions > Promoting a Knight     +2 
     9   8  ck-026    -3.879  0.5225  Specific A

E4 [exception]  Which production numbers am I not allowed to swap when I move number tokens around?
   gold: ['ck-075']   baseline: success (gold rank 1)   ->   re-ranked: success (gold rank 1)
   new was  chunk   CE logit  bi-enc  section                                  flags
     1   1  ck-075     5.577  0.5481  The Almanac > Inventor (2)                 = GOLD | ->LLM
     2  10  ck-021    -7.818  0.1975  Specific Actions > Dice Rolls             +8 ->LLM
     3   2  ck-008    -8.232  0.2980  New Game Pieces > Knights                 -1 ->LLM
     4   8  ck-093    -8.635  0.2086  The Almanac > Merchant (6)                +4 
     5   6  ck-036   -10.390  0.2306  City Improvements                         +1 
     6   9  ck-070   -10.694  0.2043  Easy Start Variant                        +3 
     7   3  ck-094   -10.970  0.2827  The Almanac > Merchant Fleet (2)          -4 
     8   4  ck-091   -11.039  0.2405  The Almanac > Commercial Harbor (2)       -4 
     9   7  ck-031   -11.22

N3 [negation/constraint]  Do all of the players have to agree before one of the optional rules is used?
   gold: ['ck-064']   baseline: success (gold rank 1)   ->   re-ranked: success (gold rank 1)
   new was  chunk   CE logit  bi-enc  section                                  flags
     1   1  ck-064     7.911  0.7922  Variants and Optional Rules                = GOLD | ->LLM
     2   3  ck-069    -5.054  0.3172  Easy Start Variant                        +1 ->LLM
     3   5  ck-070    -5.219  0.2792  Easy Start Variant                        +2 ->LLM
     4   6  ck-067    -5.291  0.2521  Nasty Variant for Experienced Players     +2 
     5   4  ck-090    -7.367  0.3100  The Almanac > Wedding (2)                 -1 
     6   8  ck-001    -8.171  0.2339  Introduction                              +2 
     7   2  ck-042    -8.789  0.3253  Progress Cards > Acquiring Progress Card  -5 
     8   7  ck-087    -9.448  0.2491  The Almanac > Saboteur (2)                -1 
     9   9  ck-063    -

O1 [other (buried detail)]  With two cities that each have a wall, how many cards can I keep when a seven comes up?
   gold: ['ck-023']   baseline: success (gold rank 1)   ->   re-ranked: success (gold rank 2)
   new was  chunk   CE logit  bi-enc  section                                  flags
     1   2  ck-074     5.561  0.6329  The Almanac > Engineer (1)                +1 ->LLM
     2   1  ck-023     4.610  0.7095  Specific Actions > City Walls             -1 GOLD | ->LLM
     3   3  ck-010     3.568  0.6302  New Game Pieces > City Walls               = ->LLM
     4   4  ck-078    -5.201  0.4314  The Almanac > Mining (2)                   = 
     5   9  ck-061    -6.487  0.4035  Attack of the Barbarians! > Losing Your   +4 
     6   7  ck-014    -7.077  0.4058  Setting Up the Game                       +1 
     7   6  ck-077    -7.094  0.4133  The Almanac > Medicine (2)                -1 
     8   5  ck-021    -7.693  0.4143  Specific Actions > Dice Rolls             -3 
     9  10 

O4 [other (buried detail)]  Can a settlement next to a pasture hex give me cloth?
   gold: ['ck-030']   baseline: retrieval stumble (gold rank 6)   ->   re-ranked: retrieval stumble (gold rank 9)
   new was  chunk   CE logit  bi-enc  section                                  flags
     1   4  ck-076    -0.455  0.3536  The Almanac > Irrigation (2)              +3 ->LLM
     2   1  ck-093    -3.249  0.3980  The Almanac > Merchant (6)                -1 ->LLM
     3   2  ck-062    -8.352  0.3710  The Merchant                              -1 ->LLM
     4   3  ck-021    -9.164  0.3608  Specific Actions > Dice Rolls             -1 
     5   9  ck-066   -10.159  0.3133  Seafarers of Catan Variant                +4 
     6  10  ck-031   -10.316  0.3122  Commodities                               +4 
     7   5  ck-020   -10.707  0.3460  Specific Actions > Dice Rolls             -2 
     8   8  ck-024   -11.020  0.3204  Specific Actions > City Walls              = 
     9   6  ck-030   -11.121  0.

In [33]:
def count(key):
    return dict(collections.Counter(r[key] for r in S5))


print("=" * 112)
print(f"STAGE 5 RESULT over N = {len(S5)} queries (every query run so far; none excluded)")
print("baseline outcomes :", count("baseline_outcome"))
print("re-ranked outcomes:", count("rerank_outcome"))
print("movement          :", count("movement"))
promoted = [r for r in S5 if r["movement"].startswith("PROMOTED")]
demoted = [r for r in S5 if r["movement"].startswith("DEMOTED")]
still = [r for r in S5 if r["movement"] == "still outside Top 3"]
print(f"PROMOTIONS (baseline stumble -> Top 3): {len(promoted)} -> {[(r['id'], r['baseline_best_gold_rank'], r['rerank_best_gold_rank']) for r in promoted] or 'none'}")
print(f"DEMOTIONS  (baseline success -> out of Top 3): {len(demoted)} -> {[(r['id'], r['baseline_best_gold_rank'], r['rerank_best_gold_rank']) for r in demoted] or 'none'}")
print(f"still outside Top 3 after re-ranking: {len(still)} -> {[(r['id'], r['baseline_best_gold_rank'], r['rerank_best_gold_rank']) for r in still] or 'none'}")
inpool = [r for r in S5 if r["baseline_best_gold_rank"]]
print(f"gold inside the Top-10 pool for {len(inpool)} of {len(S5)} queries; in the Top 3: baseline {sum(r['baseline_outcome']=='success' for r in inpool)}"
      f" -> re-ranked {sum(r['rerank_outcome']=='success' for r in inpool)}; mean best-gold rank {np.mean([r['baseline_best_gold_rank'] for r in inpool]):.2f}"
      f" -> {np.mean([r['rerank_best_gold_rank'] for r in inpool]):.2f}")
print(f"\n{'id':3} {'category':24} {'baseline':20} {'gold rank':>9}   {'re-ranked':10} {'gold rank':>9}   movement")
for r in S5:
    print(f"{r['id']:3} {r['category'][:24]:24} {r['baseline_outcome']:20} {str(r['baseline_best_gold_rank']):>9}   {r['rerank_outcome']:10} {str(r['rerank_best_gold_rank']):>9}   {r['movement']}")

(OUT_DIR / "stage5_rerank.json").write_text(json.dumps(dict(
    stage=5, pipeline="Bi-Encoder Top-10 -> Cross-Encoder re-rank -> Top-3", cross_encoder=CE_NAME, ce_max_length=CE_MAX_LEN,
    scores="raw logits (Identity activation); ties keep baseline order", corpus_sha256=FROZEN_CORPUS_SHA256, prereg_sha256=PREREG_SHA256,
    index_file_sha256=sha256_file(INDEX_PATH), k_init=K_INIT, k_final=K_FINAL, n_queries=len(S5),
    baseline_outcomes=count("baseline_outcome"), rerank_outcomes=count("rerank_outcome"), movements=count("movement"),
    queries=S5), indent=1))
assert corpus_hash(load_frozen_chunks()) == FROZEN_CORPUS_SHA256, "corpus changed during Stage 5"
print("\nsaved results/stage5_rerank.json | corpus hash re-verified: unchanged | no answer generation or timing was run")

STAGE 5 RESULT over N = 19 queries (every query run so far; none excluded)
baseline outcomes : {'success': 14, 'candidate-pool miss': 4, 'retrieval stumble': 1}
re-ranked outcomes: {'success': 14, 'candidate-pool miss': 4, 'retrieval stumble': 1}
movement          : {'kept in Top 3': 14, 'pool miss (re-ranker cannot help)': 4, 'still outside Top 3': 1}
PROMOTIONS (baseline stumble -> Top 3): 0 -> none
DEMOTIONS  (baseline success -> out of Top 3): 0 -> none
still outside Top 3 after re-ranking: 1 -> [('O4', 6, 9)]
gold inside the Top-10 pool for 15 of 19 queries; in the Top 3: baseline 14 -> re-ranked 14; mean best-gold rank 1.53 -> 1.60

id  category                 baseline             gold rank   re-ranked  gold rank   movement
Q1  candidate (Stage 2e/4)   success                      1   success            1   kept in Top 3
Q2  candidate (Stage 2e/4)   success                      1   success            1   kept in Top 3
Q3  candidate (Stage 2e/4)   success                      2  